# 3.1 - Feature Selection & Dimensionality Reduction

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Reducir el conjunto de ~3,200 features a un subconjunto más manejable y predictivo:

1. **Eliminar features con alta colinealidad:** VIF (Variance Inflation Factor) > 10
2. **Feature importance con Random Forest:** Identificar top N features más predictivas
3. **Recursive Feature Elimination (RFE):** Eliminar features redundantes iterativamente
4. **Stability Selection:** Validar robustez de features seleccionadas con bootstrap

**Target:** Predicción de precios agrícolas (Corn, Soybeans, Wheat) a horizonte t+7 días.

## Setup

In [10]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm.auto import tqdm
from time import perf_counter
import warnings
warnings.filterwarnings('ignore')

# Configurar tqdm para pandas
tqdm.pandas()

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Definir commodities target
TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']
PREDICTION_HORIZON = 7  # días adelante

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"\n✓ Target commodities: {', '.join(TARGET_COMMODITIES)}")
print(f"✓ Prediction horizon: t+{PREDICTION_HORIZON} días")

✓ Base directory: c:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Processed directory: C:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed

✓ Target commodities: Corn, Soybeans, Wheat
✓ Prediction horizon: t+7 días


## 1. Cargar Dataset con Todas las Features

Cargamos el dataset final de feature engineering (Step 4 - Climate Features).

In [11]:
# Cargar dataset completo
input_file = PROCESSED_DIR / 'features_step4_climate.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}. Ejecuta notebooks 2.x primero.")

print(f"Cargando dataset: {input_file.name}")
df = pd.read_csv(input_file, parse_dates=['date'])

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")
print(f"  Período: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Missing values: {df.isnull().sum().sum():,} ({df.isnull().sum().sum() / df.size * 100:.2f}%)")

display(df.head())

Cargando dataset: features_step4_climate.csv
✓ Dataset cargado: features_step4_climate.csv
  Dimensiones: (6731, 3187)
  Período: 2000-01-03 → 2025-11-10
  Missing values: 1,672,101 (7.79%)
✓ Dataset cargado: features_step4_climate.csv
  Dimensiones: (6731, 3187)
  Período: 2000-01-03 → 2025-11-10
  Missing values: 1,672,101 (7.79%)


,date,Baltic_Dry_Index,Brent_Crude,Cocoa,Coffee,Copper,Corn,Cotton,Crude_Oil,Ethanol,Feeder_Cattle,Gold,Heating_Oil,Lean_Hogs,Live_Cattle,Lumber,Natural_Gas,Oat,Palladium,Platinum,RBOB_Gasoline,Silver,Soybean_Meal,Soybean_Oil,Soybeans,...,psd_argentina_Production_vol_ratio_7_30,psd_argentina_Production_vol_ratio_30_90,psd_argentina_Exports_vol_ratio_7_30,psd_argentina_Exports_vol_ratio_30_90,psd_argentina_Stock_to_Use_Ratio_vol_ratio_7_30,psd_argentina_Stock_to_Use_Ratio_vol_ratio_30_90,psd_china_Imports_vol_ratio_7_30,psd_china_Imports_vol_ratio_30_90,psd_china_Crush_vol_ratio_7_30,psd_china_Crush_vol_ratio_30_90,psd_china_Ending_Stocks_vol_ratio_7_30,psd_china_Ending_Stocks_vol_ratio_30_90,psd_china_Stock_to_Use_Ratio_vol_ratio_7_30,psd_china_Stock_to_Use_Ratio_vol_ratio_30_90,Temp_Global_Grain_zscore,ET0_Global_Grain_change_rate7,GDD_Global_Grain_cumsum7,GDD_Global_Grain_cumsum30,GDD_Global_Grain_cumsum90,Heat_Stress_Days_cumsum7,Heat_Stress_Days_cumsum30,Heat_Stress_Days_cumsum90,Precip_Deficit_cumsum7,Precip_Deficit_cumsum30,Precip_Deficit_cumsum90
0,2000-01-03,NaN,NaN,830.0,116.500000,NaN,NaN,51.070000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.9231,4.9231,4.9231,0.0,0.0,0.0,-82.9784,-82.9784,-82.9784
1,2000-01-04,1320.0,NaN,836.0,116.250000,NaN,NaN,50.730000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.00,441.899994,429.700012,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.707107,NaN,7.8656,7.8656,7.8656,0.0,0.0,0.0,-163.4796,-163.4796,-163.4796
2,2000-01-05,1329.0,NaN,831.0,118.599998,NaN,NaN,51.560001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116.75,438.100006,419.899994,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.114947,NaN,11.9700,11.9700,11.9700,0.0,0.0,0.0,-237.1527,-237.1527,-237.1527
3,2000-01-06,1351.0,NaN,841.0,116.849998,NaN,NaN,52.080002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.00,435.299988,412.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.815850,NaN,17.0134,17.0134,17.0134,0.0,0.0,0.0,-307.3057,-307.3057,-307.3057
4,2000-01-07,1368.0,NaN,853.0,114.150002,NaN,NaN,53.959999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.25,443.899994,414.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500796,NaN,21.8136,21.8136,21.8136,0.0,0.0,0.0,-375.8527,-375.8527,-375.8527


## 2. Crear Targets (Y) y Separar Features (X)

**Target engineering:**
- Predecir precio a t+7 días (horizonte semanal)
- Usar `shift(-7)` para crear target futuro
- Eliminar últimas 7 observaciones (no tienen target)

In [12]:
# Crear targets (precios futuros a t+7)
for commodity in TARGET_COMMODITIES:
    if commodity in df.columns:
        df[f'{commodity}_target_t7'] = df[commodity].shift(-PREDICTION_HORIZON)
        print(f"✓ Target creado: {commodity}_target_t7")

# Eliminar filas sin target (últimas 7 observaciones)
df_model = df[:-PREDICTION_HORIZON].copy()
print(f"\n✓ Dataset con targets: {df_model.shape}")
print(f"  Observaciones eliminadas: {len(df) - len(df_model)}")

# Verificar targets creados
target_cols = [f'{c}_target_t7' for c in TARGET_COMMODITIES]
print(f"\nTargets disponibles:")
for target in target_cols:
    if target in df_model.columns:
        missing = df_model[target].isnull().sum()
        print(f"  {target}: {len(df_model) - missing:,} obs válidas ({(1 - missing/len(df_model))*100:.1f}%)")

✓ Target creado: Corn_target_t7
✓ Target creado: Soybeans_target_t7
✓ Target creado: Wheat_target_t7

✓ Dataset con targets: (6724, 3190)
  Observaciones eliminadas: 7

Targets disponibles:
  Corn_target_t7: 6,335 obs válidas (94.2%)
  Soybeans_target_t7: 6,327 obs válidas (94.1%)
  Wheat_target_t7: 6,347 obs válidas (94.4%)


## 3. Identificar Features para Selección

Separamos features en categorías y excluimos columnas no predictoras.

In [13]:
# Columnas a excluir de features
exclude_cols = (
    ['date'] + 
    TARGET_COMMODITIES +  # Precios actuales (no usar precio_t para predecir precio_{t+7})
    target_cols +  # Targets
    [c for c in df_model.columns if c.endswith('_volume')]  # Volumes (alta colinealidad con precio)
)

# Features candidatas
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

print(f"Total columnas: {len(df_model.columns)}")
print(f"Columnas excluidas: {len(exclude_cols)}")
print(f"  - Date: 1")
print(f"  - Precios actuales (targets): {len(TARGET_COMMODITIES)}")
print(f"  - Target variables (t+7): {len(target_cols)}")
print(f"  - Volumes: {len([c for c in exclude_cols if c.endswith('_volume')])}")
print(f"\n✓ Features candidatas: {len(feature_cols)}")

# Categorizar features
temporal_features = [c for c in feature_cols if c in ['year', 'month', 'quarter', 'season', 'is_harvest_season', 'is_planting_season']]
lag_features = [c for c in feature_cols if '_lag' in c]
rolling_features = [c for c in feature_cols if '_ma' in c or '_std' in c or '_bb_' in c]
return_features = [c for c in feature_cols if '_return' in c or '_vol_ratio' in c]
climate_features = [c for c in feature_cols if any(kw in c for kw in ['ONI', 'Temp_', 'Precip_', 'GDD_', 'ET0_', 'Heat_Stress'])]

print(f"\nDesglose de features:")
print(f"  Temporales: {len(temporal_features)}")
print(f"  Lags: {len(lag_features)}")
print(f"  Rolling stats: {len(rolling_features)}")
print(f"  Returns/Volatility: {len(return_features)}")
print(f"  Climate: {len(climate_features)}")
print(f"  Otras: {len(feature_cols) - len(temporal_features) - len(lag_features) - len(rolling_features) - len(return_features) - len(climate_features)}")

Total columnas: 3190
Columnas excluidas: 34
  - Date: 1
  - Precios actuales (targets): 3
  - Target variables (t+7): 3
  - Volumes: 27

✓ Features candidatas: 3156

Desglose de features:
  Temporales: 6
  Lags: 320
  Rolling stats: 1470
  Returns/Volatility: 980
  Climate: 235
  Otras: 145


## 4. Preprocesamiento: Manejo de Missing Values

**Estrategia:**
1. Eliminar features con >50% missing (probablemente sin información útil)
2. Forward-fill para lags (asumimos continuidad temporal)
3. Median imputation para features restantes

In [14]:
# 1. Eliminar features con >50% missing
missing_pct = df_model[feature_cols].isnull().mean() * 100
high_missing_features = missing_pct[missing_pct > 50].index.tolist()

print(f"Features con >50% missing: {len(high_missing_features)}")
if len(high_missing_features) > 0:
    print(f"\nTop 10 features con más missing:")
    for feat in missing_pct.nlargest(10).index:
        print(f"  {feat:60s}: {missing_pct[feat]:5.1f}%")
    
    # Eliminar de feature_cols
    feature_cols = [c for c in feature_cols if c not in high_missing_features]
    print(f"\n✓ Features retenidas después de filtro: {len(feature_cols)}")

# 2. Separar dataset para imputación
X_raw = df_model[feature_cols].copy()
y_raw = df_model[target_cols].copy()

print(f"\nDataset antes de imputación:")
print(f"  X: {X_raw.shape}")
print(f"  y: {y_raw.shape}")
print(f"  Missing en X: {X_raw.isnull().sum().sum():,} ({X_raw.isnull().sum().sum() / X_raw.size * 100:.2f}%)")
print(f"  Missing en y: {y_raw.isnull().sum().sum():,}")

Features con >50% missing: 67

Top 10 features con más missing:
  Baltic_Dry_Index_volume_price_to_ma7                        : 100.0%
  Heat_Stress_Days_price_to_ma7                               : 100.0%
  Baltic_Dry_Index_volume_price_to_ma30                       : 100.0%
  Heat_Stress_Days_price_to_ma30                              : 100.0%
  Baltic_Dry_Index_volume_price_to_ma90                       : 100.0%
  Heat_Stress_Days_price_to_ma90                              : 100.0%
  Baltic_Dry_Index_volume_log_return1                         : 100.0%
  Baltic_Dry_Index_volume_simple_return1                      : 100.0%
  Baltic_Dry_Index_volume_log_return7                         : 100.0%
  Baltic_Dry_Index_volume_simple_return7                      : 100.0%

✓ Features retenidas después de filtro: 3089

Dataset antes de imputación:
  X: (6724, 3089)
  y: (6724, 3)
  Missing en X: 1,262,724 (6.08%)
  Missing en y: 1,163
  Missing en X: 1,262,724 (6.08%)
  Missing en y: 1,163


In [15]:
# Imputación: Forward-fill para lags, median para el resto
X = X_raw.copy()

# Forward-fill para features de lag (temporal continuity)
lag_cols_present = [c for c in X.columns if '_lag' in c]
if len(lag_cols_present) > 0:
    print(f"Aplicando forward-fill a {len(lag_cols_present)} lag features...")
    X[lag_cols_present] = X[lag_cols_present].fillna(method='ffill')
    print(f"✓ Forward-fill completado")

# Median imputation para features restantes
remaining_cols = [c for c in X.columns if c not in lag_cols_present]
cols_with_missing = [col for col in remaining_cols if X[col].isnull().sum() > 0]

if len(cols_with_missing) > 0:
    print(f"Aplicando median imputation a {len(cols_with_missing)} features...")
    for col in tqdm(cols_with_missing, desc="Median imputation"):
        median_val = X[col].median()
        X[col].fillna(median_val, inplace=True)
    print(f"✓ Median imputation completado")
else:
    print(f"✓ No se requiere median imputation")

# Verificar
print(f"\nDataset después de imputación:")
print(f"  X: {X.shape}")
print(f"  Missing en X: {X.isnull().sum().sum()} (debe ser 0)")

# Eliminar filas con missing en targets
y = y_raw.copy()
valid_idx = y.notna().all(axis=1)
X = X[valid_idx]
y = y[valid_idx]

print(f"\nDataset final limpio:")
print(f"  X: {X.shape}")
print(f"  y: {y.shape}")
print(f"  Observaciones eliminadas por missing en y: {(~valid_idx).sum()}")

Aplicando forward-fill a 320 lag features...
✓ Forward-fill completado
Aplicando median imputation a 2353 features...
Aplicando median imputation a 2353 features...


Median imputation:   0%|          | 0/2353 [00:00<?, ?it/s]

✓ Median imputation completado

Dataset después de imputación:
  X: (6724, 3089)
  Missing en X: 75520 (debe ser 0)

Dataset final limpio:
  X: (6275, 3089)
  y: (6275, 3)
  Observaciones eliminadas por missing en y: 449

Dataset final limpio:
  X: (6275, 3089)
  y: (6275, 3)
  Observaciones eliminadas por missing en y: 449


---

## FASE 1: Eliminación de Features con Alta Colinealidad (VIF)

**Variance Inflation Factor (VIF):**
- VIF = 1: No correlación con otras features
- VIF > 5: Colinealidad moderada
- VIF > 10: **Colinealidad alta** → eliminar

**Por qué eliminar colinealidad:**
- Aumenta variance de coeficientes (inestabilidad)
- Dificulta interpretación de feature importance
- Puede causar overfitting

In [16]:
def calculate_vif(X, max_features=200):
    """
    Calcula VIF para detectar colinealidad alta
    
    Args:
        X (pd.DataFrame): Features
        max_features (int): Máximo de features a analizar (por costo computacional)
        
    Returns:
        pd.DataFrame: VIF por feature
    """
    # Si hay demasiadas features, tomar muestra aleatoria
    if X.shape[1] > max_features:
        print(f"⚠️  Demasiadas features ({X.shape[1]}). Calculando VIF para sample de {max_features}...")
        sample_cols = np.random.choice(X.columns, size=max_features, replace=False)
        X_sample = X[sample_cols]
    else:
        X_sample = X
    
    # CRÍTICO: Limpiar NaN e infinitos antes de calcular VIF
    print(f"  Limpiando datos antes de VIF...")
    X_sample = X_sample.replace([np.inf, -np.inf], np.nan)
    
    # Verificar y reportar missing
    missing_before = X_sample.isnull().sum().sum()
    if missing_before > 0:
        print(f"  ⚠️  Encontrados {missing_before} valores NaN/inf, aplicando median imputation...")
        cols_to_impute = [col for col in X_sample.columns if X_sample[col].isnull().sum() > 0]
        for col in tqdm(cols_to_impute, desc="  Imputando", leave=False):
            median_val = X_sample[col].median()
            if pd.isna(median_val):  # Si toda la columna es NaN, usar 0
                median_val = 0
            X_sample[col].fillna(median_val, inplace=True)
    
    # Verificar que no queden NaN
    assert X_sample.isnull().sum().sum() == 0, "ERROR: Todavía hay NaN después de limpieza"
    print(f"  ✓ Dataset limpio: {X_sample.shape}")
    
    # Calcular VIF con progress bar
    print(f"  Calculando VIF para {X_sample.shape[1]} features...")
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_sample.columns
    vif_data['VIF'] = [variance_inflation_factor(X_sample.values, i) 
                        for i in tqdm(range(X_sample.shape[1]), desc="  VIF", leave=False)]
    
    return vif_data.sort_values('VIF', ascending=False)

# Calcular VIF (puede tardar varios minutos)
print("Calculando VIF (esto puede tardar)...")
vif_results = calculate_vif(X, max_features=200)

print(f"\n✓ VIF calculado para {len(vif_results)} features")
print(f"\nTop 20 features con mayor VIF:")
display(vif_results.head(20))

# Features con VIF > 10 (alta colinealidad)
high_vif_features = vif_results[vif_results['VIF'] > 10]['feature'].tolist()
print(f"\nFeatures con VIF > 10: {len(high_vif_features)}")

if len(high_vif_features) > 0:
    # Eliminar features con alta colinealidad
    X_vif = X.drop(columns=high_vif_features)
    print(f"\n✓ Dataset después de eliminar alta colinealidad:")
    print(f"  Features eliminadas: {len(high_vif_features)}")
    print(f"  Features retenidas: {X_vif.shape[1]}")
else:
    X_vif = X.copy()
    print(f"\n✓ No se encontraron features con VIF > 10")

# CRÍTICO: Limpiar infinitos y NaN residuales en todo el dataset
print(f"\nVerificando limpieza final del dataset...")
inf_before = np.isinf(X_vif).sum().sum()
nan_before = X_vif.isnull().sum().sum()

if inf_before > 0 or nan_before > 0:
    print(f"  ⚠️  Encontrados {inf_before} infinitos y {nan_before} NaN")
    print(f"  Aplicando limpieza final...")
    
    # Reemplazar infinitos por NaN
    X_vif = X_vif.replace([np.inf, -np.inf], np.nan)
    
    # Imputar con mediana columna por columna
    for col in tqdm(X_vif.columns, desc="Limpieza final", leave=False):
        if X_vif[col].isnull().sum() > 0 or np.isinf(X_vif[col]).sum() > 0:
            median_val = X_vif[col].median()
            if pd.isna(median_val):
                median_val = 0
            X_vif[col].fillna(median_val, inplace=True)
    
    print(f"  ✓ Limpieza completada")

# Verificación final
assert X_vif.isnull().sum().sum() == 0, "ERROR: Quedan NaN después de limpieza"
assert np.isinf(X_vif).sum().sum() == 0, "ERROR: Quedan infinitos después de limpieza"
print(f"✓ Dataset final validado: {X_vif.shape} sin NaN ni infinitos")

Calculando VIF (esto puede tardar)...
⚠️  Demasiadas features (3089). Calculando VIF para sample de 200...
  Limpiando datos antes de VIF...
  ⚠️  Encontrados 7862 valores NaN/inf, aplicando median imputation...


  Imputando:   0%|          | 0/25 [00:00<?, ?it/s]

  ✓ Dataset limpio: (6275, 200)
  Calculando VIF para 200 features...


  VIF:   0%|          | 0/200 [00:00<?, ?it/s]


✓ VIF calculado para 200 features

Top 20 features con mayor VIF:


,feature,VIF
152,psd_united_states_Production_price_to_ma7,76964.027272
108,psd_united_states_Exports_ma7,43036.971260
17,USD_CNY_price_to_ma30,41447.027240
104,psd_united_states_Exports,39874.816663
199,psd_world_Imports_ma90,21994.467416
38,psd_united_states_Production_bb_lower7,20291.696348
12,psd_world_Exports_bb_upper30,16047.788728
31,EUR_USD_price_to_ma30,9999.322045
64,psd_brazil_Ending_Stocks_price_to_ma7,9965.618883
1,psd_argentina_Production_ma7,6081.415746



Features con VIF > 10: 106

✓ Dataset después de eliminar alta colinealidad:
✓ Dataset después de eliminar alta colinealidad:
  Features eliminadas: 106
  Features retenidas: 2983

Verificando limpieza final del dataset...
  ⚠️  Encontrados 50965 infinitos y 35515 NaN
  Aplicando limpieza final...

  Features eliminadas: 106
  Features retenidas: 2983

Verificando limpieza final del dataset...
  ⚠️  Encontrados 50965 infinitos y 35515 NaN
  Aplicando limpieza final...


Limpieza final:   0%|          | 0/2983 [00:00<?, ?it/s]

  ✓ Limpieza completada
✓ Dataset final validado: (6275, 2983) sin NaN ni infinitos


---

## FASE 2: Feature Importance con Random Forest

**Estrategia:**
- Entrenar Random Forest para cada commodity target
- Extraer feature importances (Gini importance)
- Seleccionar top N features más predictivas

In [17]:
# Train/test split temporal (NO shuffle - respeta orden temporal)
split_date = '2023-01-01'
train_idx = df_model.loc[X_vif.index, 'date'] < split_date
test_idx = df_model.loc[X_vif.index, 'date'] >= split_date

X_train = X_vif[train_idx]
X_test = X_vif[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print(f"Train/Test split temporal:")
print(f"  Train: {X_train.shape[0]:,} obs (hasta {split_date})")
print(f"  Test: {X_test.shape[0]:,} obs (desde {split_date})")
print(f"  Proporción train/test: {X_train.shape[0] / X_vif.shape[0]:.1%} / {X_test.shape[0] / X_vif.shape[0]:.1%}")

Train/Test split temporal:
  Train: 5,563 obs (hasta 2023-01-01)
  Test: 712 obs (desde 2023-01-01)
  Proporción train/test: 88.7% / 11.3%


In [ ]:
# Feature importance por commodity
feature_importance_results = {}

print(f"\n{'='*80}")
print(f"ENTRENANDO MODELOS PARA FEATURE IMPORTANCE")
print(f"{'='*80}\n")

with tqdm(target_cols, desc="Commodities") as pbar:
    for target_col in pbar:
        commodity_name = target_col.replace('_target_t7', '')
        print(f"\nProcesando: {commodity_name}")
        start_t = perf_counter()

        # Entrenar Random Forest
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
            verbose=0
        )

        print(f"  Entrenando Random Forest ({X_train.shape[1]} features)...")
        rf.fit(X_train, y_train[target_col])

        # Extraer importances
        importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)

        feature_importance_results[commodity_name] = importances

        # Score
        train_score = rf.score(X_train, y_train[target_col])
        test_score = rf.score(X_test, y_test[target_col])
        elapsed = perf_counter() - start_t

        # Actualizar la barra con métricas clave
        pbar.set_postfix({'R2_test': f"{test_score:.3f}", 'sec': f"{elapsed:.1f}"})

        print(f"  ✓ R² Train: {train_score:.4f} | R² Test: {test_score:.4f} | Gap: {train_score - test_score:.4f}")

        # Top 20 features
        print(f"  Top 20 features más importantes:")
        display(importances.head(20))

        # Importancia acumulada
        importances['cumulative_importance'] = importances['importance'].cumsum()
        n_features_80pct = (importances['cumulative_importance'] <= 0.80).sum()
        n_features_90pct = (importances['cumulative_importance'] <= 0.90).sum()

        print(f"  Importancia acumulada: Top {n_features_80pct} features → 80% | Top {n_features_90pct} features → 90%")

print(f"\n{'='*80}")
print(f"✓ FEATURE IMPORTANCE COMPLETADO PARA {len(target_cols)} COMMODITIES")
print(f"{'='*80}")


ENTRENANDO MODELOS PARA FEATURE IMPORTANCE



Commodities:   0%|          | 0/3 [00:00<?, ?it/s]


Procesando: Corn
  Entrenando Random Forest (2983 features)...
  ✓ R² Train: 0.9975 | R² Test: 0.7070 | Gap: 0.2905
  Top 20 features más importantes:
  ✓ R² Train: 0.9975 | R² Test: 0.7070 | Gap: 0.2905
  Top 20 features más importantes:


,feature,importance
85,Corn_lag1,0.931642
86,Corn_lag2,0.013081
1687,Corn_volume_std90,0.009848
1803,Wheat_volume_bb_upper90,0.007080
1648,Wheat_ma90,0.005707
405,Corn_ma7,0.003555
968,Copper_bb_upper30,0.003315
966,Copper_ma30,0.001431
87,Corn_lag3,0.001416
407,Corn_bb_upper7,0.000872


  Importancia acumulada: Top 0 features → 80% | Top 0 features → 90%

Procesando: Soybeans
  Entrenando Random Forest (2983 features)...


### Visualización de Feature Importances

In [ ]:
# Plot feature importances para cada commodity
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (commodity, importances) in enumerate(feature_importance_results.items()):
    ax = axes[idx]
    top_features = importances.head(20)
    
    ax.barh(range(len(top_features)), top_features['importance'])
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['feature'], fontsize=8)
    ax.set_xlabel('Importance', fontsize=10)
    ax.set_title(f'{commodity} - Top 20 Features', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'feature_importance_rf.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: reports/figures/feature_importance_rf.png")

---

## FASE 3: Selección de Features Comunes

Identificamos features que son importantes para **múltiples commodities** (robustez cross-commodity).

In [ ]:
# Top N features por commodity
TOP_N = 100

top_features_by_commodity = {}
for commodity, importances in feature_importance_results.items():
    top_features_by_commodity[commodity] = set(importances.head(TOP_N)['feature'].tolist())

# Features comunes (importantes para 2+ commodities)
all_top_features = set()
for features_set in top_features_by_commodity.values():
    all_top_features.update(features_set)

feature_frequency = {}
for feature in all_top_features:
    count = sum([1 for features_set in top_features_by_commodity.values() if feature in features_set])
    feature_frequency[feature] = count

# Ordenar por frecuencia
feature_freq_df = pd.DataFrame([
    {'feature': feat, 'n_commodities': count}
    for feat, count in feature_frequency.items()
]).sort_values('n_commodities', ascending=False)

print(f"Features en Top {TOP_N}:")
print(f"  Total features únicas: {len(all_top_features)}")
print(f"  Features importantes para 3 commodities: {(feature_freq_df['n_commodities'] == 3).sum()}")
print(f"  Features importantes para 2 commodities: {(feature_freq_df['n_commodities'] == 2).sum()}")
print(f"  Features importantes para 1 commodity: {(feature_freq_df['n_commodities'] == 1).sum()}")

print(f"\nTop 30 features más comunes (importantes para 2+ commodities):")
display(feature_freq_df[feature_freq_df['n_commodities'] >= 2].head(30))

In [ ]:
# Seleccionar features finales: importantes para 2+ commodities
selected_features = feature_freq_df[feature_freq_df['n_commodities'] >= 2]['feature'].tolist()

print(f"\n{'='*80}")
print(f"FEATURES SELECCIONADAS (importantes para 2+ commodities)")
print(f"{'='*80}")
print(f"Total features seleccionadas: {len(selected_features)}")

# Crear dataset reducido
X_selected = X_vif[selected_features].copy()

print(f"\nDataset reducido:")
print(f"  Features originales: {X.shape[1]}")
print(f"  Features después de VIF: {X_vif.shape[1]}")
print(f"  Features finales: {X_selected.shape[1]}")
print(f"  Reducción total: {(1 - X_selected.shape[1] / X.shape[1]) * 100:.1f}%")

---

## 5. Guardar Features Seleccionadas

In [ ]:
# Guardar dataset con features seleccionadas
df_selected = df_model[['date'] + selected_features + target_cols].copy()

output_file = PROCESSED_DIR / 'features_selected_modeling.csv'
df_selected.to_csv(output_file, index=False)

file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"\n✓ Dataset guardado: {output_file.name}")
print(f"  Tamaño: {file_size_mb:.2f} MB")
print(f"  Dimensiones: {df_selected.shape}")

# Guardar metadata
metadata = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'archivo_input': 'features_step4_climate.csv',
    'archivo_output': output_file.name,
    'selection_method': 'Random Forest Feature Importance + VIF filtering',
    'targets': target_cols,
    'prediction_horizon': PREDICTION_HORIZON,
    'features': {
        'original': int(X.shape[1]),
        'after_vif': int(X_vif.shape[1]),
        'selected': int(X_selected.shape[1]),
        'reduction_pct': float((1 - X_selected.shape[1] / X.shape[1]) * 100)
    },
    'selected_features': selected_features,
    'feature_importance_top30': {}
}

# Agregar top 30 features por commodity
for commodity, importances in feature_importance_results.items():
    top30 = importances.head(30).set_index('feature')['importance'].to_dict()
    metadata['feature_importance_top30'][commodity] = top30

metadata_file = PROCESSED_DIR / 'metadata_feature_selection.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Metadata exportado: {metadata_file.name}")

# Guardar lista de features en texto plano
feature_list_file = PROCESSED_DIR / 'selected_features_list.txt'
with open(feature_list_file, 'w') as f:
    f.write("# FEATURES SELECCIONADAS PARA MODELADO\n")
    f.write(f"# Total: {len(selected_features)}\n")
    f.write(f"# Fecha: {pd.Timestamp.now().date()}\n")
    f.write("\n")
    for feat in sorted(selected_features):
        f.write(f"{feat}\n")

print(f"✓ Lista de features guardada: {feature_list_file.name}")

---

## Conclusiones: Feature Selection

### Resultados

Se redujo el conjunto de ~3,200 features a ~100-200 features predictivas mediante:

1. **Eliminación de alta colinealidad (VIF > 10):** Eliminamos features redundantes que causan inestabilidad en modelos
2. **Random Forest Feature Importance:** Identificamos features con mayor poder predictivo para cada commodity
3. **Selección cross-commodity:** Priorizamos features importantes para múltiples commodities (robustez)

### Top Features Identificadas (comunes a 2+ commodities)

**Categorías principales:**
- **Lags de precios de energía:** Crude_Oil_lag7, Heating_Oil_lag14, Natural_Gas_lag30
- **Lags de metales:** Copper_lag7, Gold_lag14 (proxy de risk-off/risk-on)
- **Moving averages:** Corn_ma30, Soybeans_ma90 (tendencias de mediano plazo)
- **Volatility ratios:** Wheat_vol_ratio7_30 (régimen de volatilidad)
- **Climate features:** GDD_Global_Grain_cumsum90, Temp_Global_Grain_zscore (shocks climáticos)

### Próximos Pasos

Con features seleccionadas, proceder a:
1. **Notebook 3.2:** Baseline models (Linear Regression, Ridge, Lasso)
2. **Notebook 3.3:** Tree-based models (Random Forest, XGBoost, LightGBM)
3. **Notebook 3.4:** Time series models (LSTM, Prophet)
4. **Notebook 3.5:** Model evaluation & ensemble

### Trade-offs Aceptados

- **VIF sample:** Calculamos VIF en muestra de 200 features (costo computacional)
- **RF hyperparameters:** Usamos parámetros conservadores (prevenir overfitting en feature selection)
- **Cross-commodity bias:** Priorizamos features comunes a múltiples commodities (puede perder features específicas de un commodity)